# LeetCode 1019 — Next Greater Node In Linked List bằng Python

Link đề: https://leetcode.com/problems/next-greater-node-in-linked-list/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Linked List (danh sách liên kết), Stack (ngăn xếp), Monotonic Stack (ngăn xếp đơn điệu)

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        ...
```

Ghi chú: với **mỗi nút** của danh sách liên kết đơn, tìm giá trị của nút **đầu tiên ở phía sau** có giá trị **lớn hơn hẳn** (strictly greater); nếu không có thì ghi `0`. Trả về mảng kết quả theo thứ tự các nút. Đây là bài "next greater element" quen thuộc nhưng trên danh sách liên kết: vì chỉ duyệt được **một chiều** (từ đầu tới cuối), cách tự nhiên là **ngăn xếp đơn điệu** duyệt một lượt: ngăn xếp giữ các nút **chưa tìm được** phần tử lớn hơn; khi gặp giá trị mới lớn hơn đỉnh thì **gán đáp án cho đỉnh** rồi bỏ nó đi. Bài này **tự dựng `ListNode`** và hàm dựng/xuất danh sách để kiểm thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: "nút lớn hơn đầu tiên ở phía sau" (nghiêm ngặt) và quy ước `0` khi không có.
2. Nắm các công cụ Python cần thiết: lớp `ListNode`, hàm dựng/xuất danh sách, duyệt con trỏ `cur = cur.next`, ngăn xếp bằng `list`, `heapq` (hàng đợi ưu tiên), và gán kết quả theo **chỉ số**.
3. Hiểu nguyên lý **ngăn xếp đơn điệu**: ngăn xếp luôn có giá trị **không tăng** từ đáy lên đỉnh (các nút đang "chờ" phần tử lớn hơn); mỗi nút bị đẩy vào và bị lấy ra **tối đa một lần**, nên tổng `O(n)`.
4. Đi qua 4 cách: với mỗi nút quét phần còn lại của danh sách (brute force), chuyển ra mảng rồi duyệt **từ phải sang trái** với ngăn xếp, duyệt **một lượt trên danh sách** với ngăn xếp chỉ số (cách nộp chính), và dùng **hàng đợi ưu tiên** các nút đang chờ.
5. Tránh các lỗi kinh điển: dùng `>=` thay vì `>` (giá trị bằng nhau **không** tính là lớn hơn), dùng `if` thay `while` khi lấy ngăn xếp ra, lưu **giá trị** thay vì **chỉ số** (không biết gán đáp án cho ai), tra cứu theo giá trị bằng từ điển (giá trị trùng nhau lẫn lộn) và quên rằng nút không có phần tử lớn hơn phải là `0`.

## 2. Đề bài & ví dụ

Cho `head` là nút đầu của một danh sách liên kết gồm `n` nút.

Với **mỗi nút**, hãy tìm giá trị của **nút kế tiếp lớn hơn** (next greater node): tức giá trị của **nút đầu tiên ở phía sau** nút đó có giá trị **lớn hơn hẳn** giá trị của nó.

Trả về mảng số nguyên `answer` trong đó `answer[i]` là giá trị nút kế tiếp lớn hơn của nút thứ `i` (đánh số từ `1`). Nếu nút thứ `i` không có nút kế tiếp lớn hơn thì `answer[i] = 0`.

**Ví dụ 1:** `head = [2,1,5]` -> `[5,5,0]`.

**Ví dụ 2:** `head = [2,7,4,3,5]` -> `[7,0,5,5,0]`.

**Ràng buộc:**

- Số nút trong danh sách là `n`, `1 <= n <= 10^4`.
- `1 <= Node.val <= 10^9`.

In [1]:
from typing import List, Optional, Tuple
import heapq
import random, time

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(vals: List[int]) -> Optional[ListNode]:
    """Dựng danh sách liên kết từ list Python; trả về nút đầu (None nếu rỗng)."""
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_array(head: Optional[ListNode]) -> List[int]:
    """Duyệt danh sách và xuất ra list Python."""
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out

examples = [
    ([2, 1, 5], [5, 5, 0]),
    ([2, 7, 4, 3, 5], [7, 0, 5, 5, 0]),
]
for vals, expected in examples:
    head = build_list(vals)
    print(f"head={vals} -> dựng lại {list_to_array(head)} | kỳ vọng {expected}")
    assert list_to_array(head) == vals

head=[2, 1, 5] -> dựng lại [2, 1, 5] | kỳ vọng [5, 5, 0]
head=[2, 7, 4, 3, 5] -> dựng lại [2, 7, 4, 3, 5] | kỳ vọng [7, 0, 5, 5, 0]


## 3. Kiến thức nền

### 3.1. Danh sách liên kết đơn

Mỗi nút `ListNode` giữ `val` và con trỏ `next` tới nút kế tiếp (`None` ở cuối). Chỉ đi được **một chiều**: từ nút đầu về cuối; không truy cập theo chỉ số. Duyệt: `while cur: ...; cur = cur.next`.

In [2]:
head = build_list([2, 7, 4])
cur, steps = head, 0
while cur:
    print(f"nút {steps}: val={cur.val}, next = {None if cur.next is None else cur.next.val}")
    cur, steps = cur.next, steps + 1

nút 0: val=2, next = 7
nút 1: val=7, next = 4
nút 2: val=4, next = None


### 3.2. Ngăn xếp đơn điệu

Ta giữ một ngăn xếp các nút **chưa tìm được** phần tử lớn hơn. Quan sát then chốt: các nút trong ngăn xếp có giá trị **không tăng** từ đáy lên đỉnh. Thật vậy, nếu nút mới lớn hơn đỉnh thì đỉnh vừa tìm được đáp án (nút mới) và bị lấy ra; ta lặp tới khi đỉnh **không nhỏ hơn** nút mới rồi mới đẩy nút mới vào. Nhờ đó ngăn xếp luôn không tăng, và **mọi** nút bị lấy ra khi gặp giá trị lớn hơn đầu tiên.

In [3]:
vals = [2, 7, 4, 3, 5]
stack = []                       # các (chỉ số, giá trị) đang chờ
ans = [0] * len(vals)
for i, v in enumerate(vals):
    while stack and stack[-1][1] < v:
        j, _ = stack.pop()
        ans[j] = v
    stack.append((i, v))
    print(f"i={i} v={v}: ngăn xếp = {stack}, ans = {ans}")
assert ans == [7, 0, 5, 5, 0]

i=0 v=2: ngăn xếp = [(0, 2)], ans = [0, 0, 0, 0, 0]
i=1 v=7: ngăn xếp = [(1, 7)], ans = [7, 0, 0, 0, 0]
i=2 v=4: ngăn xếp = [(1, 7), (2, 4)], ans = [7, 0, 0, 0, 0]
i=3 v=3: ngăn xếp = [(1, 7), (2, 4), (3, 3)], ans = [7, 0, 0, 0, 0]
i=4 v=5: ngăn xếp = [(1, 7), (4, 5)], ans = [7, 0, 5, 5, 0]


### 3.3. Vì sao dùng chỉ số (không chỉ giá trị)

Đáp án phải được gán **đúng vị trí** của nút đang chờ. Nếu chỉ lưu giá trị thì không biết gán cho vị trí nào (và các giá trị trùng nhau gây lẫn lộn). Ta lưu cặp `(chỉ số, giá trị)` hoặc chỉ lưu chỉ số và tra `vals[chỉ_số]`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `[2, 7, 4, 3, 5]`.

- Nút `2`: nút đầu tiên phía sau lớn hơn `2` là `7`.
- Nút `7`: phía sau `4, 3, 5` đều nhỏ hơn: `0`.
- Nút `4`: phía sau `3` không lớn hơn, `5` lớn hơn: `5`.
- Nút `3`: phía sau `5` lớn hơn: `5`.
- Nút `5`: không có nút sau: `0`.

Kết quả `[7, 0, 5, 5, 0]`.

Duyệt một lượt bằng ngăn xếp: `2` vào; gặp `7 > 2` lấy `2` ra (đáp án `7`), `7` vào; gặp `4 < 7`: `4` vào; gặp `3 < 4`: `3` vào; gặp `5 > 3` lấy `3` ra (đáp án `5`), `5 > 4` lấy `4` ra (đáp án `5`), `5 < 7` dừng, `5` vào. Còn lại `7, 5` ở ngăn xếp: đáp án `0`.

## 5. Cách 1 — Brute Force: với mỗi nút quét phần còn lại

**Ý tưởng.** Với mỗi nút `cur`, dùng con trỏ `nxt` chạy từ `cur.next` về cuối cho tới khi gặp giá trị `> cur.val`; nếu không gặp thì `0`. Làm trực tiếp trên danh sách liên kết, đúng như định nghĩa của đề nên là **oracle độc lập** (không dùng ngăn xếp).

In [4]:
def next_greater_brute(head: Optional[ListNode]) -> List[int]:
    res = []
    cur = head
    while cur is not None:
        nxt = cur.next
        while nxt is not None and nxt.val <= cur.val:
            nxt = nxt.next
        res.append(0 if nxt is None else nxt.val)
        cur = cur.next
    return res

for vals, expected in examples:
    got = next_greater_brute(build_list(vals))
    print(f"head={vals}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

head=[2, 1, 5]: brute = [5, 5, 0] OK
head=[2, 7, 4, 3, 5]: brute = [7, 0, 5, 5, 0] OK


**Độ phức tạp và nút thắt.** `O(n^2)` tồi nhất (danh sách giảm dần: mỗi nút quét hết phần còn lại): với `n = 10^4` là `~ 5 * 10^7` bước đi con trỏ. Đo thử rồi ngoại suy:

In [5]:
for n in (500, 1000, 2000, 4000):
    head = build_list(list(range(n, 0, -1)))              # giảm dần: tồi nhất
    t0 = time.perf_counter()
    next_greater_brute(head)
    dt = time.perf_counter() - t0
    print(f"n={n:5}: {dt * 1000:8.1f} ms -> ngoại suy n=10^4: ~{dt * (10 ** 4 / n) ** 2:6.2f} s")

n=  500:      3.9 ms -> ngoại suy n=10^4: ~  1.55 s
n= 1000:     18.6 ms -> ngoại suy n=10^4: ~  1.86 s
n= 2000:     77.5 ms -> ngoại suy n=10^4: ~  1.94 s


n= 4000:    324.0 ms -> ngoại suy n=10^4: ~  2.03 s


Nút thắt: mỗi nút quét lại những phần tử mà nút trước đã quét. Nếu nút `a` đã "đi qua" `b` (vì `b <= a`) mà `b` lại cần tìm phần tử lớn hơn thì tìm tiếp từ chỗ `a` dừng: ta **tái sử dụng** thông tin bằng ngăn xếp.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ các nút "đang chờ" trong ngăn xếp; khi gặp giá trị lớn hơn thì cả loạt nút ở đỉnh nhỏ hơn đều tìm được đáp án.**

Duyệt các nút từ đầu tới cuối với ngăn xếp các cặp `(chỉ số, giá trị)`:

1. Với nút mới có giá trị `v`: **chừng nào** đỉnh ngăn xếp có giá trị `< v`, lấy đỉnh ra và đặt `answer[chỉ_số_đỉnh] = v`.
2. Thêm `answer.append(0)` cho nút mới và đẩy `(chỉ_số, v)` vào ngăn xếp.
3. Nút còn lại trong ngăn xếp sau cùng giữ đáp án `0`.

Mỗi nút vào và ra ngăn xếp tối đa một lần: `O(n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Chuyển ra mảng, duyệt từ **phải sang trái**, ngăn xếp giữ các ứng viên lớn hơn |
| 3 | Một lượt trên danh sách liên kết, ngăn xếp chỉ số (nộp) |
| 4 | Hàng đợi ưu tiên (min-heap) các nút đang chờ, `O(n log n)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mảng và duyệt từ phải sang trái

**Ý tưởng khác.** Chuyển danh sách ra mảng `vals`. Duyệt `i` từ cuối về đầu, giữ ngăn xếp chứa các **giá trị ứng viên** (đỉnh là ứng viên gần nhất, các giá trị tăng dần từ đỉnh xuống đáy). Với `vals[i]`: bỏ khỏi ngăn xếp mọi giá trị `<= vals[i]` (chúng không bao giờ là "lớn hơn hẳn" cho `vals[i]` hay cho các nút bên trái nếu `vals[i]` đã chặn); khi đó đỉnh (nếu có) là nút lớn hơn đầu tiên phía sau; rồi đẩy `vals[i]`.

**Giải thích từng đoạn.**

- Điều kiện bỏ là `stack[-1] <= vals[i]` (giá trị **bằng** cũng bỏ vì không lớn hơn hẳn và `vals[i]` đứng gần hơn nên thay thế nó).
- `res[i] = stack[-1] if stack else 0`.

In [6]:
def next_greater_right_to_left(head: Optional[ListNode]) -> List[int]:
    vals = list_to_array(head)
    n = len(vals)
    res = [0] * n
    stack: List[int] = []                      # các giá trị ứng viên, đỉnh là gần nhất
    for i in range(n - 1, -1, -1):
        while stack and stack[-1] <= vals[i]:
            stack.pop()
        res[i] = stack[-1] if stack else 0
        stack.append(vals[i])
    return res

for vals, expected in examples:
    got = next_greater_right_to_left(build_list(vals))
    print(f"head={vals}: phải sang trái = {got}", "OK" if got == expected else "SAI")
    assert got == expected

head=[2, 1, 5]: phải sang trái = [5, 5, 0] OK
head=[2, 7, 4, 3, 5]: phải sang trái = [7, 0, 5, 5, 0] OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        vals = []
        while head:
            vals.append(head.val)
            head = head.next
        n = len(vals)
        res = [0] * n
        stack = []                                  # giá trị ứng viên, đỉnh là gần nhất
        for i in range(n - 1, -1, -1):
            while stack and stack[-1] <= vals[i]:   # bằng nhau không tính là lớn hơn hẳn
                stack.pop()
            if stack:
                res[i] = stack[-1]
            stack.append(vals[i])
        return res

sol_rtl = Solution().nextLargerNodes
for vals, expected in examples:
    assert sol_rtl(build_list(vals)) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(n)` (mảng `vals` và ngăn xếp).

---

### 7.2. Cách 3 — Một lượt trên danh sách với ngăn xếp chỉ số (cách nộp chính)

**Giải thích từng đoạn.**

- `ans = []`: đáp án, thêm một phần tử `0` cho mỗi nút khi gặp nút đó.
- `stack`: các cặp `(chỉ số, giá trị)` của các nút chưa có đáp án, giá trị **không tăng** từ đáy lên đỉnh.
- Khi gặp nút giá trị `v`: `while stack and stack[-1][1] < v: idx, _ = stack.pop(); ans[idx] = v`. Điều kiện **nghiêm ngặt** `<` (giá trị bằng nhau không tính).
- Sau đó `ans.append(0)` và `stack.append((i, v))`.
- Không cần chuyển danh sách ra mảng riêng: một lượt `O(n)`.

In [8]:
def next_greater_one_pass(head: Optional[ListNode]) -> List[int]:
    ans: List[int] = []
    stack: List[Tuple[int, int]] = []
    cur, i = head, 0
    while cur is not None:
        while stack and stack[-1][1] < cur.val:
            idx, _ = stack.pop()
            ans[idx] = cur.val
        ans.append(0)
        stack.append((i, cur.val))
        cur, i = cur.next, i + 1
    return ans

for vals, expected in examples:
    got = next_greater_one_pass(build_list(vals))
    print(f"head={vals}: một lượt = {got}", "OK" if got == expected else "SAI")
    assert got == expected

head=[2, 1, 5]: một lượt = [5, 5, 0] OK
head=[2, 7, 4, 3, 5]: một lượt = [7, 0, 5, 5, 0] OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        ans = []
        stack = []                                   # (chỉ số, giá trị) các nút đang chờ nút lớn hơn
        i = 0
        while head:
            while stack and stack[-1][1] < head.val:   # nghiêm ngặt: bằng nhau không tính
                idx, _ = stack.pop()
                ans[idx] = head.val
            ans.append(0)                              # mặc định 0 nếu không có nút lớn hơn
            stack.append((i, head.val))
            head = head.next
            i += 1
        return ans

sol_one_pass = Solution().nextLargerNodes
for vals, expected in examples:
    assert sol_one_pass(build_list(vals)) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)` (mỗi nút vào và ra ngăn xếp tối đa một lần), bộ nhớ `O(n)` cho `ans` và ngăn xếp (tồi nhất khi giảm dần).

---

### 7.3. Cách 4 — Hàng đợi ưu tiên (min-heap) các nút đang chờ

**Ý tưởng khác.** Thay ngăn xếp bằng **min-heap** `(giá trị, chỉ số)` chứa các nút đang chờ. Khi gặp nút mới `v`, lấy ra mọi phần tử có giá trị `< v` (đỉnh heap là giá trị **nhỏ nhất**) và gán đáp án `v`. Đúng nhưng mỗi thao tác tốn `O(log n)`: tổng `O(n log n)`.

**Giải thích từng đoạn.**

- `heap[0][0] < v`: phần tử nhỏ nhất vẫn nhỏ hơn `v` ⇒ lấy ra.
- Vì mọi nút trong heap đều là nút đứng **trước** và chưa tìm thấy phần tử lớn hơn, gán `ans[idx] = v` đúng là nút lớn hơn **đầu tiên** (nút đầu tiên thỏa `v > giá trị` khi duyệt từ trái sang).

In [10]:
def next_greater_heap(head: Optional[ListNode]) -> List[int]:
    ans: List[int] = []
    heap: List[Tuple[int, int]] = []
    cur, i = head, 0
    while cur is not None:
        while heap and heap[0][0] < cur.val:
            _, idx = heapq.heappop(heap)
            ans[idx] = cur.val
        ans.append(0)
        heapq.heappush(heap, (cur.val, i))
        cur, i = cur.next, i + 1
    return ans

for vals, expected in examples:
    got = next_greater_heap(build_list(vals))
    print(f"head={vals}: min-heap = {got}", "OK" if got == expected else "SAI")
    assert got == expected

head=[2, 1, 5]: min-heap = [5, 5, 0] OK
head=[2, 7, 4, 3, 5]: min-heap = [7, 0, 5, 5, 0] OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        import heapq
        ans = []
        heap = []                                       # (giá trị, chỉ số) các nút đang chờ
        i = 0
        while head:
            while heap and heap[0][0] < head.val:       # mọi nút nhỏ hơn giá trị hiện tại tìm được đáp án
                _, idx = heapq.heappop(heap)
                ans[idx] = head.val
            ans.append(0)
            heapq.heappush(heap, (head.val, i))
            head = head.next
            i += 1
        return ans

sol_heap = Solution().nextLargerNodes
for vals, expected in examples:
    assert sol_heap(build_list(vals)) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n log n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `[2, 7, 4, 3, 5]`

In từng nút: các nút bị lấy ra, đáp án và ngăn xếp sau bước.

In [12]:
def trace_one_pass(vals):
    ans, stack = [], []
    for i, v in enumerate(vals):
        popped = []
        while stack and stack[-1][1] < v:
            idx, val = stack.pop()
            ans[idx] = v
            popped.append((idx, val))
        ans.append(0)
        stack.append((i, v))
        print(f"nút {i} (val={v}): lấy ra {popped or '-'} -> ans = {ans}; ngăn xếp = {stack}")
    print("đáp án cuối:", ans, "| các nút còn lại trong ngăn xếp giữ 0:", [idx for idx, _ in stack])
    return ans

assert trace_one_pass([2, 7, 4, 3, 5]) == [7, 0, 5, 5, 0]

nút 0 (val=2): lấy ra - -> ans = [0]; ngăn xếp = [(0, 2)]
nút 1 (val=7): lấy ra [(0, 2)] -> ans = [7, 0]; ngăn xếp = [(1, 7)]
nút 2 (val=4): lấy ra - -> ans = [7, 0, 0]; ngăn xếp = [(1, 7), (2, 4)]
nút 3 (val=3): lấy ra - -> ans = [7, 0, 0, 0]; ngăn xếp = [(1, 7), (2, 4), (3, 3)]
nút 4 (val=5): lấy ra [(3, 3), (2, 4)] -> ans = [7, 0, 5, 5, 0]; ngăn xếp = [(1, 7), (4, 5)]
đáp án cuối: [7, 0, 5, 5, 0] | các nút còn lại trong ngăn xếp giữ 0: [1, 4]


### 8.2. Cách 2 (phải sang trái) cho `[2, 7, 4, 3, 5]`

In ngăn xếp ứng viên khi duyệt từ cuối về đầu.

In [13]:
def trace_rtl(vals):
    n = len(vals)
    res, stack = [0] * n, []
    for i in range(n - 1, -1, -1):
        removed = []
        while stack and stack[-1] <= vals[i]:
            removed.append(stack.pop())
        res[i] = stack[-1] if stack else 0
        stack.append(vals[i])
        print(f"i={i} val={vals[i]}: bỏ {removed or '-'}, lớn hơn gần nhất = {res[i]}, ngăn xếp = {stack}")
    return res

assert trace_rtl([2, 7, 4, 3, 5]) == [7, 0, 5, 5, 0]

i=4 val=5: bỏ -, lớn hơn gần nhất = 0, ngăn xếp = [5]
i=3 val=3: bỏ -, lớn hơn gần nhất = 5, ngăn xếp = [5, 3]
i=2 val=4: bỏ [3], lớn hơn gần nhất = 5, ngăn xếp = [5, 4]
i=1 val=7: bỏ [4, 5], lớn hơn gần nhất = 0, ngăn xếp = [7]
i=0 val=2: bỏ -, lớn hơn gần nhất = 7, ngăn xếp = [7, 2]


### 8.3. Giá trị bằng nhau không được tính

`[3, 3, 5]`: nút `3` thứ nhất có nút kế tiếp `3` (bằng nhau, **không** lớn hơn hẳn) nên phải bỏ qua và lấy `5`. Cả hai nút `3` có đáp án `5`.

In [14]:
vals = [3, 3, 5]
print("đáp án đúng:", next_greater_one_pass(build_list(vals)))
assert next_greater_one_pass(build_list(vals)) == [5, 5, 0] == next_greater_brute(build_list(vals))

đáp án đúng: [5, 5, 0]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (một lượt trên danh sách với ngăn xếp chỉ số): `O(n)`, không cần mảng trung gian.

```python
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        ans = []
        stack = []                                   # (chỉ số, giá trị) các nút đang chờ nút lớn hơn
        i = 0
        while head:
            while stack and stack[-1][1] < head.val:   # nghiêm ngặt: bằng nhau không tính
                idx, _ = stack.pop()
                ans[idx] = head.val
            ans.append(0)                              # mặc định 0 nếu không có nút lớn hơn
            stack.append((i, head.val))
            head = head.next
            i += 1
        return ans
```

In [15]:
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        ans = []
        stack = []                                   # (chỉ số, giá trị) các nút đang chờ nút lớn hơn
        i = 0
        while head:
            while stack and stack[-1][1] < head.val:   # nghiêm ngặt: bằng nhau không tính
                idx, _ = stack.pop()
                ans[idx] = head.val
            ans.append(0)                              # mặc định 0 nếu không có nút lớn hơn
            stack.append((i, head.val))
            head = head.next
            i += 1
        return ans

sol = Solution()
for vals, expected in examples:
    got = sol.nextLargerNodes(build_list(vals))
    print(f"head={vals}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

head=[2, 1, 5]: [5, 5, 0] OK
head=[2, 7, 4, 3, 5]: [7, 0, 5, 5, 0] OK


## 10. Vì sao đúng

**Bất biến.** Trước khi xử lý nút thứ `i` (giá trị `v_i`): (a) mọi nút `j < i` **không** nằm trong ngăn xếp đã có `ans[j]` đúng (là nút lớn hơn đầu tiên); (b) mọi nút `j < i` **nằm** trong ngăn xếp có `ans[j] = 0` và mọi nút đứng sau nó (tới `i - 1`) đều **không lớn hơn hẳn** `v_j` (chưa tìm được nút lớn hơn); (c) ngăn xếp có giá trị **không tăng** từ đáy lên đỉnh.

**Khi xử lý nút `i`:**

- Mọi nút trong ngăn xếp có giá trị `< v_i` nhận `ans = v_i`: vì chúng chưa có nút lớn hơn trước `i` (theo (b)) và `v_i > giá trị của chúng`, nên `i` là nút lớn hơn đầu tiên. Do ngăn xếp không tăng (c), các nút có giá trị `< v_i` nằm ở **đỉnh** thành một đoạn liên tiếp: vòng `while` lấy chúng ra đúng.
- Các nút còn lại trong ngăn xếp có giá trị `>= v_i` **không** lớn hơn hẳn bởi `v_i`, giữ nguyên.
- Nút `i` được đẩy vào: giá trị `v_i <=` đỉnh mới (vì vòng `while` dừng khi đỉnh `>= v_i`), giữ (c).

**Cuối cùng**, các nút còn trong ngăn xếp không có nút lớn hơn phía sau: `ans = 0` (giá trị mặc định đã gán khi thêm nút).

**Mỗi nút vào/ra ngăn xếp tối đa một lần** ⇒ `O(n)`.

**Cách 2** là đối xứng: duyệt từ phải sang trái, ngăn xếp giữ các giá trị ứng viên; bỏ các giá trị `<= vals[i]` vì chúng không thể là đáp án của `vals[i]` (không lớn hơn hẳn) và cũng không thể là đáp án của các nút bên trái nếu `vals[i]` (đứng gần hơn và lớn hơn hoặc bằng chúng) đã thay thế; đỉnh còn lại là nút lớn hơn gần nhất.

**Cách 4** đúng vì mọi nút trong heap đều đứng trước nút mới và chưa tìm được đáp án; nút mới là nút lớn hơn đầu tiên cho mọi nút có giá trị `< v` trong heap.

**Kiểm chứng bằng máy.** Oracle (Cách 1) quét trực tiếp phần còn lại bằng con trỏ. Ta đối chiếu bốn cách trên **mọi** dãy giá trị trong `{1, 2, 3, 4}` có độ dài `<= 7` (nhiều giá trị trùng nhau) và trên nhiều danh sách ngẫu nhiên có seed.

In [16]:
from itertools import product
total = 0
for n in range(1, 8):
    for tup in product((1, 2, 3, 4), repeat=n):
        vals = list(tup)
        ref = next_greater_brute(build_list(vals))
        assert next_greater_right_to_left(build_list(vals)) == next_greater_one_pass(build_list(vals)) == next_greater_heap(build_list(vals)) == ref, vals
        assert sol.nextLargerNodes(build_list(vals)) == ref
        total += 1
print("đã đối chiếu", total, "dãy (giá trị 1..4, độ dài 1..7): các cách khớp oracle")

đã đối chiếu 21844 dãy (giá trị 1..4, độ dài 1..7): các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quét lại phần còn lại | `O(n^2)` | `O(1)` thêm |
| 2. Mảng + phải sang trái | `O(n)` | `O(n)` |
| 3. Một lượt + ngăn xếp chỉ số | `O(n)` | `O(n)` |
| 4. Min-heap | `O(n log n)` | `O(n)` |

Đo ở cỡ tối đa (`n = 10^4`) với bốn dạng: ngẫu nhiên, tăng dần, giảm dần, toàn bằng nhau:

In [17]:
rng = random.Random(11)
n = 10 ** 4
cases = {"ngẫu nhiên": [rng.randint(1, 10 ** 9) for _ in range(n)],
         "tăng dần": list(range(1, n + 1)),
         "giảm dần": list(range(n, 0, -1)),
         "toàn bằng nhau": [7] * n}
for label, vals in cases.items():
    ref = next_greater_one_pass(build_list(vals))
    row = [f"{label:16}"]
    for name, fn in (("Cách 2", sol_rtl), ("Cách 3", sol_one_pass), ("Cách 4", sol_heap), ("nộp", sol.nextLargerNodes)):
        head = build_list(vals)
        t0 = time.perf_counter()
        got = fn(head)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row))
t0 = time.perf_counter()
brute = next_greater_brute(build_list(cases["ngẫu nhiên"]))
print(f"Cách 1 (ngẫu nhiên, n = 10^4): {(time.perf_counter() - t0) * 1000:.1f} ms; khớp:", brute == next_greater_one_pass(build_list(cases["ngẫu nhiên"])))

ngẫu nhiên       | Cách 2    2.14 ms | Cách 3    1.71 ms | Cách 4    3.00 ms | nộp    1.72 ms
tăng dần         | Cách 2    1.09 ms | Cách 3    1.44 ms | Cách 4    1.45 ms | nộp    1.29 ms
giảm dần         | Cách 2    1.24 ms | Cách 3    1.46 ms | Cách 4    4.45 ms | nộp    1.42 ms
toàn bằng nhau   | Cách 2    1.59 ms | Cách 3    1.88 ms | Cách 4    2.02 ms | nộp    1.25 ms
Cách 1 (ngẫu nhiên, n = 10^4): 5.1 ms; khớp: True


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [18]:
def all_solutions():
    return {"rtl": next_greater_right_to_left, "one_pass": next_greater_one_pass, "heap": next_greater_heap,
            "sol_rtl": sol_rtl, "sol_one_pass": sol_one_pass, "sol_heap": sol_heap, "final": sol.nextLargerNodes}

def check_all(vals, expected=None, brute=True):
    results = {name: fn(build_list(vals)) for name, fn in all_solutions().items()}
    assert all(r == results["one_pass"] for r in results.values()), (vals[:15], results)
    ref = results["one_pass"]
    if brute and len(vals) <= 3000:
        assert next_greater_brute(build_list(vals)) == ref, vals[:15]
    if expected is not None:
        assert ref == expected, (vals, ref, expected)
    return ref

# 12.1 ví dụ đề
for vals, expected in examples:
    check_all(vals, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [0]),                                  # một nút: không có nút sau
    ([5, 5], [0, 0]),                            # bằng nhau không lớn hơn hẳn
    ([1, 2], [2, 0]),                            # tăng
    ([2, 1], [0, 0]),                            # giảm
    ([3, 3, 5], [5, 5, 0]),                      # bằng nhau rồi lớn hơn
    ([1, 3, 1, 5], [3, 5, 5, 0]),                # hai giá trị 1 có đáp án khác nhau
    ([5, 4, 3, 2, 1], [0, 0, 0, 0, 0]),          # giảm dần: không có
    ([1, 2, 3, 4, 5], [2, 3, 4, 5, 0]),          # tăng dần: nút kế tiếp
    ([1, 7, 5, 1, 9, 2, 5, 1], [7, 9, 9, 9, 0, 5, 0, 0]),   # ví dụ kinh điển (tính tay ở bài tập 2)
    ([2, 7, 4, 3, 5], [7, 0, 5, 5, 0]),          # ví dụ 2
    ([10 ** 9, 1, 10 ** 9], [0, 10 ** 9, 0]),    # giá trị lớn nhất; bằng nhau ở hai đầu
    ([4, 4, 4, 4], [0, 0, 0, 0]),                # toàn bằng nhau
    ([2, 1, 2, 1, 2, 3], [3, 2, 3, 2, 3, 0]),    # xen kẽ rồi tăng
]
for vals, expected in edge:
    check_all(vals, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm, so với oracle brute)
def wrong_non_strict(vals):                       # dùng <= thay vì < khi lấy ngăn xếp ra: coi bằng nhau là lớn hơn
    ans, stack = [], []
    for i, v in enumerate(vals):
        while stack and stack[-1][1] <= v:
            idx, _ = stack.pop()
            ans[idx] = v
        ans.append(0)
        stack.append((i, v))
    return ans

def wrong_if_not_while(vals):                     # if thay while: chỉ lấy ra một phần tử
    ans, stack = [], []
    for i, v in enumerate(vals):
        if stack and stack[-1][1] < v:
            idx, _ = stack.pop()
            ans[idx] = v
        ans.append(0)
        stack.append((i, v))
    return ans

def wrong_dict_by_value(vals):                    # tra cứu theo giá trị: các giá trị trùng nhau bị lẫn
    nxt, stack = {}, []
    for v in vals:
        while stack and stack[-1] < v:
            nxt[stack.pop()] = v
        stack.append(v)
    return [nxt.get(v, 0) for v in vals]

def wrong_default_none(vals):                     # đáp án mặc định None thay vì 0 (đề yêu cầu 0)
    ans, stack = [], []
    for i, v in enumerate(vals):
        while stack and stack[-1][1] < v:
            idx, _ = stack.pop()
            ans[idx] = v
        ans.append(None)
        stack.append((i, v))
    return ans

def wrong_rtl_keep_equal(vals):                   # phải sang trái nhưng chỉ bỏ giá trị < vals[i] (giữ giá trị bằng)
    n = len(vals)
    res, stack = [0] * n, []
    for i in range(n - 1, -1, -1):
        while stack and stack[-1] < vals[i]:
            stack.pop()
        res[i] = stack[-1] if stack else 0
        stack.append(vals[i])
    return res

def find_counter(wrong, vmax=4, max_len=6):
    for n in range(1, max_len + 1):
        for tup in product(range(1, vmax + 1), repeat=n):
            vals = list(tup)
            if wrong(vals) != next_greater_brute(build_list(vals)):
                return vals
    return None

counters = {}
for name, fn in (("dùng <= (coi bằng nhau là lớn hơn)", wrong_non_strict), ("if thay while", wrong_if_not_while),
                 ("tra cứu theo giá trị", wrong_dict_by_value), ("mặc định None thay vì 0", wrong_default_none),
                 ("phải sang trái giữ giá trị bằng", wrong_rtl_keep_equal)):
    vals = find_counter(fn)
    assert vals is not None, name
    counters[name] = vals
    print(f"{name:36} vals={vals}: sai -> {fn(vals)}, đúng -> {next_greater_brute(build_list(vals))}")
    check_all(vals)
print("12.3 phản ví dụ: OK")

dùng <= (coi bằng nhau là lớn hơn)   vals=[1, 1]: sai -> [1, 0], đúng -> [0, 0]
if thay while                        vals=[1, 1, 2]: sai -> [0, 2, 0], đúng -> [2, 2, 0]
tra cứu theo giá trị                 vals=[1, 2, 1]: sai -> [2, 0, 2], đúng -> [2, 0, 0]
mặc định None thay vì 0              vals=[1]: sai -> [None], đúng -> [0]
phải sang trái giữ giá trị bằng      vals=[1, 1]: sai -> [1, 0], đúng -> [0, 0]
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên: nhiều miền giá trị (nhiều trùng nhau và toàn phân biệt), n tới 3000 so với oracle; n = 10^4 so các cách nhanh
rng = random.Random(4019)
for _ in range(800):
    n = rng.randint(1, 40)
    vmax = rng.choice([2, 5, 10, 10 ** 9])
    check_all([rng.randint(1, vmax) for _ in range(n)])
for _ in range(10):
    n = rng.randint(500, 3000)
    check_all([rng.randint(1, rng.choice([5, 100, 10 ** 9])) for _ in range(n)])
for _ in range(5):
    n = 10 ** 4
    check_all([rng.randint(1, rng.choice([5, 100, 10 ** 9])) for _ in range(n)], brute=False)
print("12.4 ngẫu nhiên: OK")

# 12.5 tính chất: đáp án 0 hoặc một giá trị có trong danh sách và > giá trị nút; nút cuối luôn 0; giá trị lớn nhất (đầu tiên xuất hiện) luôn 0
rng = random.Random(5019)
for _ in range(400):
    vals = [rng.randint(1, 20) for _ in range(rng.randint(1, 40))]
    ans = sol.nextLargerNodes(build_list(vals))
    assert len(ans) == len(vals) and ans[-1] == 0
    for i, (v, a) in enumerate(zip(vals, ans)):
        assert a == 0 or (a > v and a in vals[i + 1:])
        assert (a == 0) == all(x <= v for x in vals[i + 1:])
    assert all(a == 0 for v, a in zip(vals, ans) if v == max(vals))
print("12.5 tính chất: OK")

# 12.6 không sửa danh sách đầu vào
vals = [rng.randint(1, 50) for _ in range(100)]
head = build_list(vals)
sol.nextLargerNodes(head)
assert list_to_array(head) == vals
print("12.6 danh sách đầu vào không bị sửa: OK")

12.4 ngẫu nhiên: OK
12.5 tính chất: OK
12.6 danh sách đầu vào không bị sửa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<=` thay vì `<` (coi giá trị bằng nhau là lớn hơn)

Đề yêu cầu **lớn hơn hẳn**. Dùng `<=` làm giá trị bằng nhau "tìm thấy" nhau.

In [21]:
vals = counters["dùng <= (coi bằng nhau là lớn hơn)"]
print("vals =", vals, "| sai ->", wrong_non_strict(vals), "| đúng ->", Solution().nextLargerNodes(build_list(vals)))
assert wrong_non_strict(vals) != Solution().nextLargerNodes(build_list(vals))
print("[3, 3, 5]: sai", wrong_non_strict([3, 3, 5]), "(đúng [5, 5, 0])")

vals = [1, 1] | sai -> [1, 0] | đúng -> [0, 0]
[3, 3, 5]: sai [3, 5, 0] (đúng [5, 5, 0])


Vì sao sai: nút `3` thứ nhất sẽ nhận đáp án `3` (nút kế tiếp bằng nó). **Đúng:** điều kiện lấy ra là `stack[-1][1] < v` (nghiêm ngặt).

### Lỗi 2 — Dùng `if` thay vì `while`

Một giá trị mới có thể lớn hơn **nhiều** nút ở đỉnh ngăn xếp; chỉ lấy một nút làm các nút còn lại mất đáp án.

In [22]:
vals = counters["if thay while"]
print("vals =", vals, "| sai ->", wrong_if_not_while(vals), "| đúng ->", Solution().nextLargerNodes(build_list(vals)))
assert wrong_if_not_while(vals) != Solution().nextLargerNodes(build_list(vals))

vals = [1, 1, 2] | sai -> [0, 2, 0] | đúng -> [2, 2, 0]


Vì sao sai: ví dụ 2 (`[2,7,4,3,5]`): `5` phải lấy ra cả `3` và `4`. **Đúng:** `while stack and stack[-1][1] < v`.

### Lỗi 3 — Tra cứu theo giá trị

Lưu kết quả vào từ điển khóa là **giá trị** làm các nút trùng giá trị đè lên nhau.

In [23]:
vals = counters["tra cứu theo giá trị"]
print("vals =", vals, "| sai ->", wrong_dict_by_value(vals), "| đúng ->", Solution().nextLargerNodes(build_list(vals)))
assert wrong_dict_by_value(vals) != Solution().nextLargerNodes(build_list(vals))
print("[1, 3, 1, 5]: sai", wrong_dict_by_value([1, 3, 1, 5]), "(đúng [3, 5, 5, 0])")

vals = [1, 2, 1] | sai -> [2, 0, 2] | đúng -> [2, 0, 0]
[1, 3, 1, 5]: sai [5, 5, 5, 0] (đúng [3, 5, 5, 0])


Vì sao sai: hai nút giá trị `1` có đáp án khác nhau (`3` và `5`) nhưng chung một khóa. **Đúng:** gán theo **chỉ số** (hoặc theo nút).

### Lỗi 4 — Mặc định `None` thay vì `0`

Nút không có phần tử lớn hơn phải là `0`.

In [24]:
vals = counters["mặc định None thay vì 0"]
print("vals =", vals, "| sai ->", wrong_default_none(vals), "| đúng ->", Solution().nextLargerNodes(build_list(vals)))
assert wrong_default_none(vals) != Solution().nextLargerNodes(build_list(vals))

vals = [1] | sai -> [None] | đúng -> [0]


Vì sao sai: đề: "set answer[i] = 0". **Đúng:** `ans.append(0)` khi thêm nút.

### Lỗi 5 — Phải sang trái nhưng giữ giá trị bằng

Ở cách duyệt từ phải sang trái, phải bỏ cả giá trị **bằng** `vals[i]` (không lớn hơn hẳn); nếu chỉ bỏ giá trị nhỏ hơn thì nhận nhầm giá trị bằng làm đáp án.

In [25]:
vals = counters["phải sang trái giữ giá trị bằng"]
print("vals =", vals, "| sai ->", wrong_rtl_keep_equal(vals), "| đúng ->", Solution().nextLargerNodes(build_list(vals)))
assert wrong_rtl_keep_equal(vals) != Solution().nextLargerNodes(build_list(vals))

vals = [1, 1] | sai -> [1, 0] | đúng -> [0, 0]


Vì sao sai: `stack[-1] < vals[i]` giữ lại giá trị bằng `vals[i]` ở đỉnh nên `res[i]` nhận nó. **Đúng:** `stack[-1] <= vals[i]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "phần tử lớn hơn/nhỏ hơn đầu tiên ở phía sau (hoặc phía trước)" ⇒ **ngăn xếp đơn điệu**, `O(n)`.
- Duyệt **trái sang phải**: ngăn xếp giữ các phần tử **chờ đáp án**, khi gặp giá trị lớn hơn thì **phát đáp án** cho cả loạt ở đỉnh (`while`, nghiêm ngặt).
- Duyệt **phải sang trái**: ngăn xếp giữ các **ứng viên**, bỏ những ứng viên `<=` phần tử hiện tại.
- Danh sách liên kết chỉ duyệt một chiều: dùng cách trái sang phải trực tiếp trên danh sách, hoặc chuyển ra mảng.
- Lưu **chỉ số** (hoặc cặp chỉ số-giá trị) để gán đáp án đúng chỗ.
- Cùng họ: Next Greater Element I/II (496, 503), Daily Temperatures (739), Largest Rectangle in Histogram (84), Remove Nodes From Linked List (2487).
- Test nên có: một nút, toàn bằng nhau, tăng dần, giảm dần, giá trị trùng với đáp án khác nhau, `n = 10^4`.

## 15. Biến thể

1. **Danh sách vòng** (nút cuối nối về nút đầu): mỗi nút tìm nút lớn hơn đầu tiên theo vòng (bài 503): duyệt hai lần.
2. **Next smaller / previous greater:** đổi chiều so sánh hoặc chiều duyệt.
3. **Khoảng cách** tới nút lớn hơn đầu tiên (bài 739, bài tập 5).
4. Bài liên quan: 496, 503, 739, 2487.

Biến thể 1 (danh sách vòng, giả sử nhiều nút có thể bằng nhau): duyệt mảng nhân đôi, kiểm bằng oracle vét cạn.

In [26]:
def next_greater_circular(vals: List[int]) -> List[int]:
    n = len(vals)
    ans = [0] * n
    stack: List[int] = []
    for i in range(2 * n):
        v = vals[i % n]
        while stack and vals[stack[-1]] < v:
            ans[stack.pop()] = v
        if i < n:
            stack.append(i)
    return ans

def next_greater_circular_brute(vals):
    n = len(vals)
    out = []
    for i in range(n):
        res = 0
        for k in range(1, n):
            if vals[(i + k) % n] > vals[i]:
                res = vals[(i + k) % n]
                break
        out.append(res)
    return out

rng = random.Random(15)
for _ in range(400):
    vals = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    assert next_greater_circular(vals) == next_greater_circular_brute(vals), vals
print(next_greater_circular([1, 2, 1]), "(ví dụ của bài 503: [2, 0, 2] với giá trị 0 thay -1)")
assert next_greater_circular([1, 2, 1]) == [2, 0, 2]

[2, 0, 2] (ví dụ của bài 503: [2, 0, 2] với giá trị 0 thay -1)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại | Từng nút tìm tiếp | `O(n^2)` | `O(1)` | Oracle |
| 2. Mảng + phải sang trái | Ứng viên gần nhất | `O(n)` | `O(n)` | Khi đã có mảng |
| 3. Một lượt + ngăn xếp chỉ số | Phát đáp án khi gặp lớn hơn | `O(n)` | `O(n)` | **Cách nộp chính** |
| 4. Min-heap | Heap các nút đang chờ | `O(n log n)` | `O(n)` | Cách nhìn khác |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
head = build_list([3, 1, 2])
print(list_to_array(head), head.next.val, head.next.next.next)
print(Solution().nextLargerNodes(build_list([3, 1, 2])), Solution().nextLargerNodes(build_list([2, 2])))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba giá trị
my_guess_2 = None      # hai danh sách

### Bài 2 — Tính tay (dễ-vừa)

`head = [1, 7, 5, 1, 9, 2, 5, 1]`. Ghi mảng đáp án (nút kế tiếp lớn hơn của từng nút).

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách một lượt.

In [29]:
def exercise_3(head: Optional[ListNode]) -> List[int]:
    ans, stack = [], []
    i = 0
    while head:
        # TODO 1: while ngăn xếp không rỗng và giá trị ở đỉnh < head.val: lấy ra và gán ans[chỉ số] = head.val
        # TODO 2: thêm 0 vào ans cho nút hiện tại
        # TODO 3: đẩy (i, head.val) vào ngăn xếp; di chuyển head và i
        pass
    return ans


# for vals, expected in examples:
#     assert exercise_3(build_list(vals)) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(head)` bằng cách chuyển ra mảng rồi duyệt **từ phải sang trái**.

In [30]:
def exercise_4(head: Optional[ListNode]) -> List[int]:
    # TODO
    pass


# for vals, expected in examples:
#     assert exercise_4(build_list(vals)) == expected

### Bài 5 — Mini challenge: khoảng cách tới nút lớn hơn (khó)

Viết `exercise_5(head)` trả về, với mỗi nút, **số bước** (số cạnh) từ nút đó tới nút lớn hơn hẳn đầu tiên ở phía sau, hoặc `0` nếu không có.

```python
exercise_5(build_list([2, 1, 5]))        -> [2, 1, 0]
exercise_5(build_list([2, 7, 4, 3, 5]))  -> [1, 0, 2, 1, 0]
exercise_5(build_list([3, 3, 3]))        -> [0, 0, 0]
```
Gợi ý: như Cách 3 nhưng gán `i - idx` thay vì giá trị.

In [31]:
def exercise_5(head: Optional[ListNode]) -> List[int]:
    # TODO
    pass


# assert exercise_5(build_list([2, 1, 5])) == [2, 1, 0]
# assert exercise_5(build_list([2, 7, 4, 3, 5])) == [1, 0, 2, 1, 0]
# assert exercise_5(build_list([3, 3, 3])) == [0, 0, 0]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `list_to_array` duyệt cả danh sách; `head.next.val` là giá trị nút thứ hai.
- Mức 2: danh sách `[3, 1, 2]`: nút thứ hai có `val = 1`; sau nút thứ ba là `None`.
- Mức 3: dòng 1: `[3, 1, 2] 1 None`; dòng 2: `[0, 2, 0] [0, 0]`.

**Bài 2**
- Mức 1: nút `1`: phía sau `7` lớn hơn; nút `7`: phía sau có `9` (đầu tiên lớn hơn `7`).
- Mức 2: nút `5`: phía sau `1`, `9`: `9`; nút `1` (thứ tư): `9`; nút `9`: không có; nút `2`: `5`; nút `5`: phía sau chỉ có `1`: không có; nút `1` cuối: không có.
- Mức 3: `[7, 9, 9, 9, 0, 5, 0, 0]`.

**Bài 3**
- Mức 1: `while stack and stack[-1][1] < head.val:`.
- Mức 2: `idx, _ = stack.pop(); ans[idx] = head.val`.
- Mức 3: `ans.append(0); stack.append((i, head.val)); head = head.next; i += 1`.

**Bài 4**
- Mức 1: `vals = list_to_array(head)`; `res = [0] * n`.
- Mức 2: với `i` từ `n - 1` về `0`: bỏ các giá trị `<= vals[i]` khỏi ngăn xếp.
- Mức 3: `res[i] = stack[-1] if stack else 0`; `stack.append(vals[i])`.

**Bài 5**
- Mức 1: ngăn xếp chỉ số như Cách 3.
- Mức 2: khi lấy `idx` ra tại chỉ số `i`: `ans[idx] = i - idx`.
- Mức 3: các nút còn lại giữ `0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
head = build_list([3, 1, 2])
print(list_to_array(head), head.next.val, head.next.next.next)
print(Solution().nextLargerNodes(build_list([3, 1, 2])), Solution().nextLargerNodes(build_list([2, 2])))

assert (list_to_array(head), head.next.val, head.next.next.next) == ([3, 1, 2], 1, None)
assert (Solution().nextLargerNodes(build_list([3, 1, 2])), Solution().nextLargerNodes(build_list([2, 2]))) == ([0, 2, 0], [0, 0])

[3, 1, 2] 1 None
[0, 2, 0] [0, 0]


### Đáp án Bài 2

`[1, 7, 5, 1, 9, 2, 5, 1]`: `1 -> 7`, `7 -> 9`, `5 -> 9`, `1 -> 9`, `9 -> 0`, `2 -> 5`, `5 -> 0`, `1 -> 0`.

In [33]:
hand_answer_solution = [7, 9, 9, 9, 0, 5, 0, 0]
assert Solution().nextLargerNodes(build_list([1, 7, 5, 1, 9, 2, 5, 1])) == hand_answer_solution
assert next_greater_brute(build_list([1, 7, 5, 1, 9, 2, 5, 1])) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(head: Optional[ListNode]) -> List[int]:
    ans, stack = [], []
    i = 0
    while head:
        while stack and stack[-1][1] < head.val:
            idx, _ = stack.pop()
            ans[idx] = head.val
        ans.append(0)
        stack.append((i, head.val))
        head = head.next
        i += 1
    return ans

for vals, expected in examples:
    assert exercise_3_solution(build_list(vals)) == expected
rng = random.Random(3)
for _ in range(300):
    vals = [rng.randint(1, 10) for _ in range(rng.randint(1, 20))]
    assert exercise_3_solution(build_list(vals)) == next_greater_brute(build_list(vals))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(head: Optional[ListNode]) -> List[int]:
    vals = list_to_array(head)
    n = len(vals)
    res, stack = [0] * n, []
    for i in range(n - 1, -1, -1):
        while stack and stack[-1] <= vals[i]:
            stack.pop()
        res[i] = stack[-1] if stack else 0
        stack.append(vals[i])
    return res

for vals, expected in examples:
    assert exercise_4_solution(build_list(vals)) == expected
rng = random.Random(4)
for _ in range(300):
    vals = [rng.randint(1, 10) for _ in range(rng.randint(1, 20))]
    assert exercise_4_solution(build_list(vals)) == next_greater_brute(build_list(vals))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(head: Optional[ListNode]) -> List[int]:
    ans, stack = [], []
    i = 0
    while head:
        while stack and stack[-1][1] < head.val:
            idx, _ = stack.pop()
            ans[idx] = i - idx
        ans.append(0)
        stack.append((i, head.val))
        head = head.next
        i += 1
    return ans

def distance_brute(vals):
    out = []
    for i, v in enumerate(vals):
        d = 0
        for j in range(i + 1, len(vals)):
            if vals[j] > v:
                d = j - i
                break
        out.append(d)
    return out

assert exercise_5_solution(build_list([2, 1, 5])) == [2, 1, 0]
assert exercise_5_solution(build_list([2, 7, 4, 3, 5])) == [1, 0, 2, 1, 0]
assert exercise_5_solution(build_list([3, 3, 3])) == [0, 0, 0]
rng = random.Random(5)
for _ in range(300):
    vals = [rng.randint(1, 10) for _ in range(rng.randint(1, 20))]
    assert exercise_5_solution(build_list(vals)) == distance_brute(vals)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp đơn điệu một lượt)

```python
class Solution:
    def nextLargerNodes(self, head: Optional[ListNode]) -> List[int]:
        ans = []
        stack = []                                   # (chỉ số, giá trị) các nút đang chờ nút lớn hơn
        i = 0
        while head:
            while stack and stack[-1][1] < head.val:   # nghiêm ngặt: bằng nhau không tính
                idx, _ = stack.pop()
                ans[idx] = head.val
            ans.append(0)                              # mặc định 0
            stack.append((i, head.val))
            head = head.next
            i += 1
        return ans
```

Công thức nhớ nhanh: duyệt trái sang phải; gặp giá trị lớn hơn thì **phát đáp án** cho mọi nút nhỏ hơn ở đỉnh ngăn xếp; còn lại giữ `0`.

### Checklist trước khi submit

- [ ] So sánh **nghiêm ngặt** (`<`), giá trị bằng nhau không tính?
- [ ] Dùng `while` (không `if`) để lấy nhiều nút ra một lần?
- [ ] Lưu **chỉ số** để gán đáp án đúng nút (không tra cứu theo giá trị)?
- [ ] Mỗi nút thêm đáp án mặc định `0`?
- [ ] Nếu duyệt phải sang trái: bỏ các giá trị `<=` phần tử hiện tại?
- [ ] Đã thử: một nút, toàn bằng nhau, tăng dần, giảm dần, giá trị trùng với đáp án khác nhau, `n = 10^4`?